# DART Data Assimilation with CESM-MOM6

This tutorial walks through setting up and running ensemble data assimilation using
[DART](https://dart.ucar.edu) (Data Assimilation Research Testbed) with a regional MOM6
ocean model in CESM. DART is the **ESP** (External System Processing) component of CESM —
it is built and run as a standard CESM component, not as a separate tool.

A typical DART–CESM workflow consists of three main steps:

1. **Clone, create, and build** a CESM case configured for data assimilation.
2. **Prepare observations** in DART `obs_seq` format using `dartobsgen`.
3. **Submit** the case — CESM calls DART automatically each assimilation cycle.

**Prerequisites**: Access to an HPC system like Derecho (or equivalent) and a CROCODILE
CESM environment. Follow the
[CrocoDash Getting Started](../tutorials/crocodash_tutorial.ipynb) tutorial first if needed.

# CESM DA Installation

DART is integrated as the ESP component of CESM. You will clone the CROCODILE fork
of CESM, create a case with a DA-enabled compset, set a few XML variables, and build.

### Clone the CESM Repository and checkout the DA branch.

Use the CROCODILE fork of CESM on the `dart-cesm3.0-alphabranch` branch, then
populate the sub-components with `git-fleximod`:

```bash
git clone https://github.com/hkershaw-brown/CESM.git CESM_DA
cd CESM_DA/
git checkout dart-cesm3.0-alphabranch
./bin/git-fleximod update
```

# CrocoDash Installation

1. Clone the repository from GitHub

```bash
git clone --recurse-submodules https://github.com/CROCODILE-CESM/CrocoDash.git
```

2. Create the environment (called CrocoDash by default) using the provided environment.yml file
```bash
cd CrocoDash
conda env create -f environment.yml 
```

3. Activate the environment
```bash
conda activate CrocoDash
pytest tests/test_installation.py
```

# SECTION 1:Generate a regional MOM6 domain

### Step 1.1: Horizontal Grid

In [ ]:

from CrocoDash.grid import Grid

grid = Grid(
  resolution = 0.05, # in degrees
  xstart = 278.0, # min longitude in [0, 360]
  lenx = 3.0, # longitude extent in degrees
  ystart = 7.0, # min latitude in [-90, 90]
  leny = 3.0, # latitude extent in degrees
  name = "panama1",
)

### Step 1.2: Topography

In [ ]:
from CrocoDash.topo import Topo

topo = Topo(
    grid = grid,
    min_depth = 9.5, # in meters
)

In [ ]:
from pathlib import Path

bathymetry_path= Path("<GEBCO>")
# The GEBCO bathymetry path on Derecho is: <GEBCO>

if not bathymetry_path.exists():
    raise FileNotFoundError("Bathymetry file not found, please replace with path to bathymetry file")

topo.set_from_dataset(
    bathymetry_path = bathymetry_path,
    longitude_coordinate_name="lon",
    latitude_coordinate_name="lat",
    vertical_coordinate_name="elevation"
)

In [ ]:
topo.depth.plot()

In [ ]:
%matplotlib ipympl
from CrocoDash.topo_editor import TopoEditor
TopoEditor(topo)

### Step 1.3: Vertical Grid

In [ ]:
from CrocoDash.vgrid import VGrid

vgrid  = VGrid.hyperbolic(
    nk = 75, # number of vertical levels
    depth = topo.max_depth,
    ratio=20.0 # target ratio of top to bottom layer thicknesses
)

In [ ]:
import matplotlib.pyplot as plt
plt.close()
# Create the plot
for depth in vgrid.z:
    plt.axhline(y=depth, linestyle='-')  # Horizontal lines

plt.ylim(max(vgrid.z) + 10, min(vgrid.z) - 10)  # Invert y-axis so deeper values go down
plt.ylabel("Depth")
plt.title("Vertical Grid")
plt.show()

## SECTION 2: Create the CESM case

After generating the MOM6 domain, the next step is to create a CESM case using CrocoDash. This process is straightforward and involves instantiating the CrocoDash Case object. The Case object requires the following inputs:

* CESM Source Directory: A local path to a compatible CESM source copy.
* Case Name: A unique name for the CESM case.
* Input Directory: The directory where all necessary input files will be written.
* MOM6 Domain Objects: The horizontal grid, topography, and vertical grid created in the previous section.
* Project ID: (Optional) A project ID, if required by the machine.
* Compset: The set of models to be used in the Case. Standalone Ocean, Ocean-BGC, Ocean-Seaice, Ocean-Runoff
* Number of instances: This is set the number of ensemble members

### Step 2.1: Specify case name and directories:

Begin by specifying the case name and the necessary directory paths. Ensure the CESM root directory points to your own local copy of CESM. Below is an example setup:

In [ ]:
from pathlib import Path
# CESM case (experiment) name
casename = "panama-not"

# CESM source root (Update this path accordingly!!!)
cesmroot ="<DART_CESMROOT>"

# Place where all your input files go 
inputdir = Path("<DART_WORKDIR>") / "croc_input" / casename
    
# CESM case directory
caseroot = Path("<DART_WORKDIR>") / "croc_cases" / casename

### Step 2.2: Create the Case

To create the CESM case, instantiate the `Case` object as shown below. This will automatically set up the CESM case based on the provided inputs: The `cesmroot` argument specifies the path to your local CESM source directory. The `caseroot` argument defines the directory where the case will be created. CrocoDash will handle all necessary namelist modifications and XML changes to align with the MOM6 domain objects generated earlier.

In [ ]:
from CrocoDash.case import Case

case = Case(
    cesmroot = cesmroot,
    caseroot = caseroot,
    inputdir = inputdir,
    ocn_grid = grid,
    ocn_vgrid = vgrid,
    ocn_topo = topo,
    project = 'CESM0030',
    override = True,
    machine = "derecho",
    compset = "GR_JRA" # This is the alias of the compset, the longname (which is printed when you run this command) is 1850_DATM%JRA_SLND_SICE_MOM6%REGIONAL_SROF_SGLC_SWAV. Feel free to use either way!
)

## SECTION 2: Create the CESM case

```
casedir = "<DART_WORKDIR>/cases/da-mom6-test.0001"
```

The compset `G_JRA_DA` is the DA-enabled ocean compset. Use `--ninst` to set the ensemble
size and `--multi-driver` to run all members in a single job.

> **Note** `RUN_STARTDATE` must match the period covered by your observation sequence files.

```
./cime/scripts/create_newcase \
    --run-unsupported \
    --res TL319_t232 \
    --compset G_JRA_DA \
    --case $casedir \
    --ninst 3 \
    --multi-driver \
    --project <PROJECT_CODE>
```

## Step 1.3: Configure and build

```bash
cd $casedir
./case.setup

./xmlchange CALENDAR=GREGORIAN
./xmlchange DATA_ASSIMILATION_OCN=TRUE
./xmlchange RUN_STARTDATE=2013-04-01   # must match observation sequence files

./case.setup --reset
./case.build
```

To inspect all DA-related XML variables after setup:

```bash
./xmlquery --partial DATA_ASS   # DATA_ASSIMILATION_* flags
./xmlquery --partial ESP        # confirms DART is the ESP component
```

## Step 1.4: Inspect and edit DART namelists

After `case.build`, `preview_namelists` writes a resolved `input.nml` to:

```
Buildconf/dartconf/input.nml
```

The list of observation sequence files expected by DART is in:

```
Buildconf/dart.input_data_list
```

To customize DART settings, edit `user_nl_dart`

```bash
vim user_nl_dart
```

Key namelist groups for MOM6 DA:
- `&filter_nml` — inflation, localization half-widths
- `&model_nml` — MOM6 state vector variables (temperature, salinity, SSH, …)
- `&obs_kind_nml` — which observation types are assimilated vs. evaluated only

# SECTION 2: Prepare Observations with _dartobsgen_

DART ingests observations through `obs_seq` files — one file per assimilation window.
`dartobsgen` is a pip-installable Python package that generates non-overlapping `obs_seq`
files from CrocoLake, the CROCODILE observational database.

```bash
cd /path/to/dartobsgen
pip install -e .
```

## Step 2.1: Configure obs generation

Set the time period, spatial domain, observation types, and assimilation frequency.
The period must overlap with `RUN_STARTDATE` set in the case.

Set the Source to `crocolake` to generate `obs_seq` files from CrocoLake observations.
Supported observation types include Argo T/S, glider T/S, and bottle measurements
from GLODAP — sourced from CrocoLake. See the `dartobsgen` [README](https://github.com/CROCODILE-CESM/dartobsgen) for the full list.


In [ ]:
import datetime
from dartobsgen import ObsGenConfig, CrocLakeSource, generate_obs_sequences

config = ObsGenConfig(
    start=datetime.datetime(2013, 4, 1),   # match RUN_STARTDATE
    end=datetime.datetime(2013, 4, 8),
    lat_min=5,    lat_max=60,
    lon_min=-100, lon_max=-30,
    obs_types=["ARGO_TEMPERATURE", "ARGO_SALINITY"],
    assimilation_frequency=datetime.timedelta(hours=6),
    output_dir="./obs_output",
)

source = CrocLakeSource(
    crocolake_path="<CROCOLAKE_PATH>",
    dart_path="<DART_PATH>",
)

## Step 2.2: Generate obs_seq files

Running `generate_obs_sequences` creates one `obs_seq` file per assimilation window according configuration and observation source. 
Files are named `obs_seq.YYYY-MM-DD-{seconds-of-day}.out` by default, matching
DART's standard naming convention.

Windows are half-open `[t0, t0 + freq)` — no observation appears in two windows.

In [ ]:
written_files = generate_obs_sequences(config, source, max_workers=None)
print(written_files)

## Step 2.3: (Optional) Trim observations to the model domain

Use `trim_obs_seq` to remove observations outside your regional MOM6 domain.
You can build the polygon from explicit vertices, a NetCDF boundary file, or
a 2D land/sea mask.

In [ ]:
from dartobsgen import trim_obs_seq, polygon_from_netcdf_mask

# Build domain polygon from the MOM6 ocean mask
poly = polygon_from_netcdf_mask(
    "<CESM_CASE>/ocean_mask.nc",
    mask_var="mask",
    lat_var="lat",
    lon_var="lon",
)

# Trim all generated obs_seq files to the domain
for path in written_files:
    kept = trim_obs_seq(path, poly)
    print(f"  {path.name}: {'kept' if kept else 'empty — removed'}")

# SECTION 3: Submit the Case

Once the case is built and obs_seq files are in place, submit with the standard
CESM command `./case.submit`. CESM will call DART automatically at the end of each assimilation
window — no separate submission step is needed.

```bash
cd $casedir
./case.submit
```

CESM–DART cycling:
1. CESM advances all `--ninst` ensemble members in parallel (forecast step).
2. At the end of the window, CESM's ESP layer calls DART `filter`.
3. `filter` reads the obs_seq file for that window, assimilates, and writes
   updated restart files for all members.
4. CESM reads the updated restarts and advances to the next window.

# SECTION 4: Examine Assimilation Output

After `filter` runs each cycle, DART writes diagnostic NetCDF files to the run directory:

| File | Contents |
|---|---|
| `obs_seq.final` | Observation-space prior/posterior values |
| `preassim_mean.nc` | Ensemble mean ocean state **before** assimilation |
| `postassim_mean.nc` | Ensemble mean ocean state **after** assimilation |
| `output_mean.nc` | Updated ensemble mean in CESM restart form |
| `output_sd.nc` | Updated ensemble spread in CESM restart form |

todo case name and timestamps in file names

## Step 4.1: Observation-space diagnostics

`obs_seq.final` contains the forward operator prior and optionally posterior values and information on whether the observation was assimilated for each observation.  [pyDARTdiags](https://ncar.github.io/pyDARTdiags/) can be used to examine the results, binning by observation type and model level.

In [ ]:
import pydartdiags.obs_sequence.obs_sequence as obsq
from pydartdiags.stats import stats

obs_seq = obsq.ObsSequence(data_file)

used_obs = obs_seq.select_used_qcs()
stats.diag_stats(used_obs)
stats.grand_statistics(used_obs)



## Step 4.2: State-space increments

The increment (posterior − prior ensemble mean) shows where and how much the
observations are correcting the model state each cycle.

In [ ]:
preassim  = xr.open_dataset(f"{run_dir}/preassim_mean.nc")
postassim = xr.open_dataset(f"{run_dir}/postassim_mean.nc")

# Sea surface temperature increment at the first output time
increment = (
    postassim["Temp"].isel(Time=0, zl=0)
    - preassim["Temp"].isel(Time=0, zl=0)
)
increment.plot(cmap="RdBu_r", robust=True)
plt.title("SST increment (posterior − prior)")
plt.show()

In [ ]:
preassim = xr.open_dataset("<DART_OUTPUT>/preassim_mean.nc")
postassim = xr.open_dataset("<DART_OUTPUT>/postassim_mean.nc")

# Sea surface temperature increment
increment = postassim["Temp"].isel(Time=0, zl=0) - preassim["Temp"].isel(Time=0, zl=0)
increment.plot(cmap="RdBu_r", robust=True)
plt.title("SST increment (posterior − prior)")
plt.show()